# 🏦 Desenvolvimento de IA para Análise Preditiva [T1 e T2]
## Situação de Aprendizagem (Projeto Avaliativo) — Módulo 2 - Semana 13

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
![Python](https://img.shields.io/badge/Python-3.10%2B-blue)
![FastAPI](https://img.shields.io/badge/FastAPI-0.115.0-009688)
![Pydantic](https://img.shields.io/badge/Pydantic-2.9.2-E92063)
![Scikit-Learn](https://img.shields.io/badge/Scikit--Learn-1.5.2-F7931E)
![Streamlit](https://img.shields.io/badge/Streamlit-1.39.0-FF4B4B)

---

### 🎯 Objetivo do Notebook (Compatível com Google Colab e Ambiente Local)
Este **Jupyter Notebook (`.ipynb`)** implementa ponta a ponta o **Microsserviço Preditivo de Análise de Risco e Concessão de Crédito**, cumprindo rigorosamente o item **5.1 do Roteiro da Aplicação** (*"aceito como notebook (.ipynb) ou Python puro (.py), organizado com cabeçalhos em Markdown estruturando todas as Fases e Desafios, justificando as tomadas de decisões técnicas"*).

Ao executar as células deste notebook no **Google Colab** (ou localmente), você irá:
1. **Configurar o Ambiente e Dependências (`requirements.txt`)**: Instalar versões fixadas de `fastapi`, `uvicorn`, `pydantic`, `scikit-learn`, `streamlit`, `plotly`, `httpx` e `pytest`.
2. **Fase 1 — Preparação e Serialização do Modelo (Pipeline Offline)**: Gerar/carregar a base em caminho relativo (`data/credito_dataset.csv`), dividir estratificadamente em **Treino (70%)**, **Validação (15%)** e **Teste (15%)**, treinar o `Pipeline` do Scikit-Learn (`SimpleImputer` + `StandardScaler` + `OneHotEncoder` + `RandomForestClassifier`), exibir gráficos de avaliação e serializar o artefato em `models/modelo_emprestimo.joblib`.
3. **Fase 2 — Estrutura Modular da API FastAPI**: Criar automaticamente os pacotes Python em camadas (`app/schemas/`, `app/services/`, `app/routers/` e `main.py`).
4. **Fase 3 — Ciclo de Vida e Inicialização Otimizada (`Lifespan`)**: Implementar `@asynccontextmanager` para carregar o modelo **uma única vez na memória RAM** no startup da aplicação.
5. **Fase 4 — Modelagem e Validação Estrita de Dados (`Pydantic`)**: Definir `DadosClienteInput`, `PredicaoCreditoOutput` e `HealthResponse` com restrições `Field(ge, le, gt, min_length)`.
6. **Fase 5 — Implementação de Endpoints, Tratamento HTTP e Testes (`Pytest`)**: Testar ao vivo `GET /health`, `POST /predict` e `POST /v1/predict` comprovando os status `200 OK`, `422 Unprocessable Entity` e `400 Bad Request`.
7. **Dashboard Executivo em Streamlit + Execução em Background no Colab**: Subir o servidor `Uvicorn` e o `Streamlit` diretamente no notebook.
8. **Automação GitFlow e Versionamento GitHub (Seção 5.3)**: Executar o fluxo completo de branches (`main`, `develop` e `feature/*`) diretamente pelo notebook.

---
## ⚙️ Etapa 0: Configuração do Ambiente, Variáveis e Dependências (`requirements.txt`)

**Justificativa Técnica (Requisito 5.1 — Reprodutibilidade e Caminhos Relativos):**
- Criamos a estrutura de diretórios usando exclusivamente `pathlib.Path` com caminhos relativos (`data/`, `models/`, `app/schemas/`, `app/services/`, `app/routers/`, `tests/`).
- Geramos o arquivo `requirements.txt` com versões fixadas e instalamos os pacotes necessários no ambiente do Google Colab.

In [ ]:
from pathlib import Path
import os

# Definicao de caminhos 100% relativos (Proibido caminhos absolutos locais - Item 5.1)
BASE_DIR = Path(".").resolve()
for pasta in [
    "data",
    "models",
    "app/schemas",
    "app/services",
    "app/routers",
    "tests",
]:
    (BASE_DIR / pasta).mkdir(parents=True, exist_ok=True)

# Criacao dos arquivos __init__.py dos pacotes modulares
(BASE_DIR / "app/__init__.py").write_text(
    '"""Pacote principal da API FastAPI para Analise Preditiva de Credito."""\n',
    encoding="utf-8",
)

print(f"[OK] Estrutura modular criada em caminho relativo a partir de: {BASE_DIR.name}/")

In [ ]:
%%writefile requirements.txt
# Dependencias da API Web e Validacao
fastapi==0.115.0
uvicorn[standard]==0.30.6
pydantic==2.9.2

# Dependencias de Ciencia de Dados e Machine Learning
pandas==2.2.3
numpy==2.1.1
scikit-learn==1.5.2
joblib==1.4.2

# Dashboard Interativo e Visualizacao de Dados
streamlit==1.39.0
plotly==5.24.1

# Testes e Cliente HTTP
httpx==0.27.2
pytest==8.3.3

In [ ]:
# Instala as dependencias no Google Colab / Jupyter de forma silenciosa
!pip install -q -r requirements.txt
print("[SUCESSO] Todas as bibliotecas do requirements.txt estao instaladas e prontas!")

---
## 🧪 Fase 1: Preparação e Serialização do Modelo (Pipeline Offline)

### Justificativas das Tomadas de Decisões Técnicas:
1. **Prevenção de *Data Leakage* (Vazamento de Dados)**:
   - A base `data/credito_dataset.csv` (`3.000` clientes) é dividida de forma **estratificada (`stratify=y`)** em **70% Treino (`2.100`)**, **15% Validação (`450`)** e **15% Teste (`450`)** *antes* de qualquer cálculo estatístico.
2. **Pipeline Integrado (`Pipeline` + `ColumnTransformer`)**:
   - **Atributos Numéricos** (`idade`, `pontuacao_credito`, `renda_mensal`, `valor_emprestimo`, `tempo_emprego_anos`, `inadimplencia_previa`): Imputação de nulos pela mediana (`SimpleImputer(strategy="median")`) para robustez contra *outliers* de renda, seguida de padronização `StandardScaler()`.
   - **Atributos Categóricos** (`tipo_moradia`, `finalidade_emprestimo`): Imputação pela moda (`SimpleImputer(strategy="most_frequent")`) seguida de `OneHotEncoder(handle_unknown="ignore")`.
   - **Estimador**: `RandomForestClassifier` com `200` árvores, `max_depth=10` e `class_weight="balanced"`.
3. **Serialização do Artefato Único (`models/modelo_emprestimo.joblib`)**:
   - Empacotamos o `Pipeline` completo junto com metadados de versão e métricas de teste em um único binário `.joblib`.

In [ ]:
%%writefile train_pipeline.py
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Caminhos relativos (Item 5.1: Proibido caminhos absolutos da maquina)
BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
DATASET_PATH = DATA_DIR / "credito_dataset.csv"
MODEL_PATH = MODELS_DIR / "modelo_emprestimo.joblib"
METRICS_PATH = MODELS_DIR / "relatorio_metricas.json"


def gerar_ou_carregar_dados(
    caminho_csv: Path,
    n_amostras: int = 3000,
) -> pd.DataFrame:
    """Carrega o CSV local ou gera a base realista de Risco de Credito."""
    caminho_csv.parent.mkdir(parents=True, exist_ok=True)
    if caminho_csv.exists():
        return pd.read_csv(caminho_csv)

    rng = np.random.default_rng(seed=42)
    idade = rng.integers(18, 75, size=n_amostras).astype(float)
    pontuacao_credito = np.clip(
        rng.normal(620, 110, size=n_amostras), 300, 850
    ).round().astype(float)
    renda_mensal = np.clip(
        rng.lognormal(mean=8.3, sigma=0.55, size=n_amostras), 1200, 50000
    ).round(2)
    valor_emprestimo = np.clip(
        rng.lognormal(mean=9.4, sigma=0.7, size=n_amostras), 1000, 150000
    ).round(2)
    tempo_emprego_anos = np.clip(
        rng.exponential(scale=5.0, size=n_amostras), 0, 40
    ).round(1)
    inadimplencia_previa = rng.choice(
        [0, 1], size=n_amostras, p=[0.82, 0.18]
    )
    tipo_moradia = rng.choice(
        ["PROPRIA", "ALUGADA", "FINANCIADA", "OUTROS"],
        size=n_amostras,
        p=[0.40, 0.35, 0.20, 0.05],
    )
    finalidade_emprestimo = rng.choice(
        ["EDUCACAO", "SAUDE", "NEGOCIOS", "PESSOAL", "QUITACAO_DIVIDAS", "IMOVEL"],
        size=n_amostras,
        p=[0.18, 0.12, 0.20, 0.22, 0.18, 0.10],
    )
    comprometimento = valor_emprestimo / (renda_mensal * 12.0)
    score_latente = (
        0.018 * (pontuacao_credito - 600)
        + 0.015 * (idade - 30)
        + 0.08 * tempo_emprego_anos
        - 3.8 * comprometimento
        - 2.1 * inadimplencia_previa
        + np.where(tipo_moradia == "PROPRIA", 0.55, 0.0)
        + np.where(tipo_moradia == "ALUGADA", -0.35, 0.0)
        + np.where(finalidade_emprestimo == "QUITACAO_DIVIDAS", -0.45, 0.2)
        + rng.normal(0, 0.65, size=n_amostras)
    )
    prob_aprovacao = 1 / (1 + np.exp(-score_latente))
    aprovado = (prob_aprovacao >= 0.5).astype(int)
    df = pd.DataFrame({
        "idade": idade,
        "pontuacao_credito": pontuacao_credito,
        "renda_mensal": renda_mensal,
        "valor_emprestimo": valor_emprestimo,
        "tempo_emprego_anos": tempo_emprego_anos,
        "inadimplencia_previa": inadimplencia_previa,
        "tipo_moradia": tipo_moradia,
        "finalidade_emprestimo": finalidade_emprestimo,
        "aprovado": aprovado,
    })
    for col in ["renda_mensal", "tempo_emprego_anos", "tipo_moradia"]:
        mask_nulos = rng.random(n_amostras) < 0.03
        df.loc[mask_nulos, col] = np.nan
    df.to_csv(caminho_csv, index=False)
    return df


def construir_pipeline() -> Pipeline:
    """Constroi o Pipeline de Pre-processamento + Classificacao."""
    colunas_numericas = [
        "idade", "pontuacao_credito", "renda_mensal",
        "valor_emprestimo", "tempo_emprego_anos", "inadimplencia_previa",
    ]
    colunas_categoricas = ["tipo_moradia", "finalidade_emprestimo"]
    transformador_numerico = Pipeline(steps=[
        ("imputador_mediana", SimpleImputer(strategy="median")),
        ("escalonador_padrao", StandardScaler()),
    ])
    transformador_categorico = Pipeline(steps=[
        ("imputador_moda", SimpleImputer(strategy="most_frequent")),
        ("codificador_onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    preprocessador = ColumnTransformer(transformers=[
        ("num", transformador_numerico, colunas_numericas),
        ("cat", transformador_categorico, colunas_categoricas),
    ])
    return Pipeline(steps=[
        ("preprocessamento", preprocessador),
        ("classificador", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_split=5,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1,
        )),
    ])


def avaliar_particao(modelo: Pipeline, x_p: pd.DataFrame, y_p: pd.Series, nome: str) -> dict:
    y_pred = modelo.predict(x_p)
    y_prob = modelo.predict_proba(x_p)[:, 1]
    return {
        "particao": nome,
        "amostras": int(len(y_p)),
        "acuracia": round(float(accuracy_score(y_p, y_pred)), 4),
        "precisao": round(float(precision_score(y_p, y_pred)), 4),
        "recall": round(float(recall_score(y_p, y_pred)), 4),
        "f1_score": round(float(f1_score(y_p, y_pred)), 4),
        "roc_auc": round(float(roc_auc_score(y_p, y_prob)), 4),
        "matriz_confusao": confusion_matrix(y_p, y_pred).tolist(),
    }


def executar_treinamento() -> None:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    df = gerar_ou_carregar_dados(DATASET_PATH)
    X = df.drop(columns=["aprovado"])
    y = df["aprovado"]
    X_treino, X_temp, y_treino, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    X_val, X_teste, y_val, y_teste = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )
    pipeline = construir_pipeline()
    pipeline.fit(X_treino, y_treino)
    m_treino = avaliar_particao(pipeline, X_treino, y_treino, "treino")
    m_val = avaliar_particao(pipeline, X_val, y_val, "validacao")
    m_teste = avaliar_particao(pipeline, X_teste, y_teste, "teste")
    print("=== RELATORIO DE CLASSIFICACAO (CONJUNTO DE TESTE - 15%) ===")
    print(classification_report(y_teste, pipeline.predict(X_teste), target_names=["Reprovado (0)", "Aprovado (1)"]))
    artefato = {
        "pipeline": pipeline,
        "versao_modelo": "1.0.0",
        "algoritmo": "RandomForestClassifier (Scikit-Learn Pipeline)",
        "features_esperadas": list(X.columns),
        "metricas_teste": m_teste,
    }
    joblib.dump(artefato, MODEL_PATH)
    relatorio = {"versao_modelo": "1.0.0", "metricas": {"treino": m_treino, "validacao": m_val, "teste": m_teste}}
    METRICS_PATH.write_text(json.dumps(relatorio, indent=4, ensure_ascii=False), encoding="utf-8")
    print(f"[SUCESSO] Modelo salvo em: {MODEL_PATH.relative_to(BASE_DIR)}")
    print(f"[SUCESSO] Metricas salvas em: {METRICS_PATH.relative_to(BASE_DIR)}")


if __name__ == "__main__":
    executar_treinamento()

In [ ]:
# Executa o treinamento da Fase 1 e exibe os graficos interativos de metricas no Notebook
import json
import pandas as pd
import plotly.express as px
from train_pipeline import executar_treinamento, METRICS_PATH

executar_treinamento()

relatorio = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
df_metricas = pd.DataFrame(list(relatorio["metricas"].values()))
display(df_metricas[["particao", "amostras", "acuracia", "precisao", "recall", "f1_score", "roc_auc"]])

# Grafico de Matriz de Confusao Interativo
mc = relatorio["metricas"]["teste"]["matriz_confusao"]
fig_cm = px.imshow(
    mc,
    text_auto=True,
    x=["Reprovado (0)", "Aprovado (1)"],
    y=["Reprovado (0)", "Aprovado (1)"],
    color_continuous_scale="Blues",
    title="Fase 1: Matriz de Confusao no Conjunto de Teste (15%)",
)
fig_cm.show()

---
## 🛡️ Fase 4: Modelagem e Validação Estrita de Dados (`Pydantic`)

### Justificativas Técnicas (`app/schemas/cliente.py`):
- **Evolução do Arquivo Base (`DadosCliente`)**: Expandimos a classe original para `DadosClienteInput` (mantendo o alias `DadosCliente = DadosClienteInput`), adicionando restrições estritas via `Field(ge=..., le=..., gt=..., min_length=..., max_length=...)` e `Enum`.
- **Prevenção de Dados Inválidos (`HTTP 422 Unprocessable Entity`)**: Qualquer tentativa de enviar idade menor que 18 anos (`ge=18`), score fora de `300..850` ou strings em campos numéricos é bloqueada automaticamente pelo Pydantic antes de chegar ao modelo.
- **Contratos Explícitos de Saída (`PredicaoCreditoOutput` e `HealthResponse`)**: Garantem tipagem forte na resposta JSON e documentação completa no Swagger (`/docs`).

In [ ]:
%%writefile app/schemas/cliente.py
from enum import Enum
from typing import Optional
from pydantic import BaseModel, ConfigDict, Field


class TipoMoradiaEnum(str, Enum):
    """Categorias validas de situacao habitacional."""
    PROPRIA = "PROPRIA"
    ALUGADA = "ALUGADA"
    FINANCIADA = "FINANCIADA"
    OUTROS = "OUTROS"


class FinalidadeEmprestimoEnum(str, Enum):
    """Finalidades validas de solicitacao de credito."""
    EDUCACAO = "EDUCACAO"
    SAUDE = "SAUDE"
    NEGOCIOS = "NEGOCIOS"
    PESSOAL = "PESSOAL"
    QUITACAO_DIVIDAS = "QUITACAO_DIVIDAS"
    IMOVEL = "IMOVEL"


class DadosClienteInput(BaseModel):
    """Schema de Entrada validado pelo Pydantic (Gera HTTP 422 se invalido)."""
    nome_cliente: Optional[str] = Field(
        default="Cliente Padrao",
        min_length=2,
        max_length=80,
        description="Nome do solicitante (2 a 80 caracteres).",
    )
    idade: int = Field(
        ...,
        ge=18,
        le=100,
        description="Idade do cliente em anos completos (18 a 100).",
    )
    pontuacao_credito: int = Field(
        ...,
        ge=300,
        le=850,
        description="Score de credito (faixa entre 300 e 850).",
    )
    renda_mensal: float = Field(
        default=5000.0,
        gt=0.0,
        le=1000000.0,
        description="Renda mensal bruta comprovada (R$ > 0).",
    )
    valor_emprestimo: float = Field(
        default=12000.0,
        ge=500.0,
        le=500000.0,
        description="Valor solicitado para emprestimo (R$ 500 a R$ 500.000).",
    )
    tempo_emprego_anos: float = Field(
        default=3.0,
        ge=0.0,
        le=60.0,
        description="Tempo de emprego atual em anos (0 a 60).",
    )
    inadimplencia_previa: int = Field(
        default=0,
        ge=0,
        le=1,
        description="Historico de inadimplencia previa (0 = Nao, 1 = Sim).",
    )
    tipo_moradia: TipoMoradiaEnum = Field(default=TipoMoradiaEnum.PROPRIA)
    finalidade_emprestimo: FinalidadeEmprestimoEnum = Field(
        default=FinalidadeEmprestimoEnum.PESSOAL
    )
    model_config = ConfigDict(str_strip_whitespace=True, extra="forbid")


# Alias mantendo o nome exato do arquivo cliente.py original
DadosCliente = DadosClienteInput


class PredicaoCreditoOutput(BaseModel):
    """Schema Estrito de Saida da Predicao de Credito."""
    nome_cliente: Optional[str] = Field(
        default=None,
        description="Identificador do solicitante avaliado.",
    )
    idade_informada: int = Field(
        ..., ge=18, le=100, description="Idade recebida na requisicao."
    )
    pontuacao_informada: int = Field(
        ..., ge=300, le=850, description="Score recebido na requisicao."
    )
    decisao: str = Field(
        ..., description="Status de negocio: 'Aprovado' ou 'Reprovado'."
    )
    codigo_decisao: int = Field(
        ..., ge=0, le=1, description="Classe predita (1 = Aprovado, 0 = Reprovado)."
    )
    probabilidade_aprovacao: float = Field(
        ..., ge=0.0, le=1.0, description="Probabilidade estimada de aprovacao."
    )
    probabilidade_risco_inadimplencia: float = Field(
        ..., ge=0.0, le=1.0, description="Probabilidade complementar de risco."
    )
    faixa_risco: str = Field(
        ..., description="Faixa ('Baixo Risco', 'Risco Moderado', 'Alto Risco')."
    )
    indice_comprometimento_anual: float = Field(
        ..., ge=0.0, description="Razao entre emprestimo e renda anual."
    )
    versao_modelo: str = Field(
        ..., description="Versao do modelo em RAM que gerou a predicao."
    )


class HealthResponse(BaseModel):
    """Schema de Saida do endpoint de monitoramento GET /health."""
    status_servico: str = Field(
        ..., description="Estado operacional ('online' ou 'degradado')."
    )
    modelo_carregado_em_ram: bool = Field(
        ..., description="True se o modelo reside ativamente na memoria RAM."
    )
    algoritmo: Optional[str] = Field(
        default=None, description="Identificacao do pipeline carregado."
    )
    versao_modelo: Optional[str] = Field(
        default=None, description="Versao do modelo carregado no Lifespan."
    )
    metricas_referencia_teste: Optional[dict] = Field(
        default=None, description="Metricas de teste registradas no artefato."
    )

In [ ]:
%%writefile app/schemas/__init__.py
from app.schemas.cliente import (
    DadosCliente,
    DadosClienteInput,
    FinalidadeEmprestimoEnum,
    HealthResponse,
    PredicaoCreditoOutput,
    TipoMoradiaEnum,
)

__all__ = [
    "DadosCliente",
    "DadosClienteInput",
    "FinalidadeEmprestimoEnum",
    "HealthResponse",
    "PredicaoCreditoOutput",
    "TipoMoradiaEnum",
]

---
## 🧠 Fases 2 e 3: Camada de Serviço Desacoplada e Modelo em Memória RAM (`app/services/credito_service.py`)

### Justificativas Técnicas:
- **Desacoplamento em Camadas (Fase 2)**: Nenhuma lógica matemática ou chamada ao Scikit-Learn ocorre dentro dos arquivos de rota HTTP.
- **Cumprimento da Proibição Técnica (Fase 3)**: O método `carregar_modelo_em_ram()` lê o arquivo `.joblib` **uma única vez** quando invocado pelo `lifespan` na inicialização da aplicação. O método `realizar_predicao()` executa `self._pipeline.predict()` diretamente na **memória RAM**, garantindo latência de ~2ms e zero gargalo de I/O de disco.

In [ ]:
%%writefile app/services/credito_service.py
from pathlib import Path
from typing import Any, Dict, Optional
import joblib
import pandas as pd

from app.schemas.cliente import DadosClienteInput, PredicaoCreditoOutput

BASE_DIR = Path(__file__).resolve().parent.parent.parent
DEFAULT_MODEL_PATH = BASE_DIR / "models" / "modelo_emprestimo.joblib"


class RegraNegocioException(ValueError):
    """Excecao para violacao semantica de regra de negocio (HTTP 400)."""


class ModeloNaoCarregadoException(RuntimeError):
    """Excecao caso o modelo nao esteja carregado em RAM (HTTP 500)."""


class CreditoModelService:
    """Servico Singleton que mantem o Pipeline Scikit-Learn em memoria RAM."""

    def __init__(self, model_path: Path = DEFAULT_MODEL_PATH) -> None:
        self.model_path: Path = model_path
        self._pipeline: Optional[Any] = None
        self._versao_modelo: str = "desconhecida"
        self._algoritmo: str = "desconhecido"
        self._metricas_teste: Dict[str, Any] = {}

    @property
    def esta_carregado(self) -> bool:
        """Retorna True se o pipeline preditivo reside na memoria RAM."""
        return self._pipeline is not None

    def carregar_modelo_em_ram(self) -> None:
        """Carrega o arquivo .joblib na RAM (chamado 1 unica vez no startup)."""
        if not self.model_path.exists():
            raise FileNotFoundError(
                f"Artefato nao encontrado em '{self.model_path}'."
            )
        artefato = joblib.load(self.model_path)
        if isinstance(artefato, dict) and "pipeline" in artefato:
            self._pipeline = artefato["pipeline"]
            self._versao_modelo = artefato.get("versao_modelo", "1.0.0")
            self._algoritmo = artefato.get(
                "algoritmo", type(self._pipeline).__name__
            )
            self._metricas_teste = artefato.get("metricas_teste", {})
        else:
            self._pipeline = artefato
            self._versao_modelo = "1.0.0-legado"
            self._algoritmo = type(artefato).__name__
            self._metricas_teste = {}

    def descarregar_modelo_da_ram(self) -> None:
        """Libera a referencia da memoria RAM no encerramento da API."""
        self._pipeline = None

    def obter_status_saude(self) -> Dict[str, Any]:
        """Retorna os metadados operacionais para o endpoint GET /health."""
        return {
            "status_servico": "online" if self.esta_carregado else "degradado",
            "modelo_carregado_em_ram": self.esta_carregado,
            "algoritmo": self._algoritmo if self.esta_carregado else None,
            "versao_modelo": self._versao_modelo if self.esta_carregado else None,
            "metricas_referencia_teste": (
                self._metricas_teste if self.esta_carregado else None
            ),
        }

    def realizar_predicao(
        self, cliente: DadosClienteInput
    ) -> PredicaoCreditoOutput:
        """Executa a inferencia em memoria RAM a partir do schema validado."""
        if not self.esta_carregado or self._pipeline is None:
            raise ModeloNaoCarregadoException(
                "O modelo preditivo nao esta carregado na memoria RAM."
            )

        # 1. Regra de Negocio: Tempo de emprego vs. Idade laboral (HTTP 400)
        idade_laboral_maxima = max(0, cliente.idade - 14)
        if cliente.tempo_emprego_anos > idade_laboral_maxima:
            raise RegraNegocioException(
                f"Inconsistencia cadastral: tempo_emprego_anos "
                f"({cliente.tempo_emprego_anos}) excede a idade laboral "
                f"possivel ({idade_laboral_maxima} anos)."
            )

        # 2. Regra de Negocio: Comprometimento maximo de 20x a renda anual
        renda_anual = cliente.renda_mensal * 12.0
        indice_comprometimento = cliente.valor_emprestimo / renda_anual
        if indice_comprometimento > 20.0:
            raise RegraNegocioException(
                "Valor solicitado excede 20x a renda anual declarada."
            )

        dados_entrada = pd.DataFrame([{
            "idade": float(cliente.idade),
            "pontuacao_credito": float(cliente.pontuacao_credito),
            "renda_mensal": float(cliente.renda_mensal),
            "valor_emprestimo": float(cliente.valor_emprestimo),
            "tempo_emprego_anos": float(cliente.tempo_emprego_anos),
            "inadimplencia_previa": int(cliente.inadimplencia_previa),
            "tipo_moradia": cliente.tipo_moradia.value,
            "finalidade_emprestimo": cliente.finalidade_emprestimo.value,
        }])

        # Predicao em Memoria RAM (Zero leitura de disco!):
        resultado_classe = int(self._pipeline.predict(dados_entrada)[0])
        probabilidades = self._pipeline.predict_proba(dados_entrada)[0]
        prob_aprovacao = float(probabilidades[1])
        prob_risco = float(probabilidades[0])
        status_decisao = "Aprovado" if resultado_classe == 1 else "Reprovado"

        if prob_aprovacao >= 0.75:
            faixa_risco = "Baixo Risco"
        elif prob_aprovacao >= 0.45:
            faixa_risco = "Risco Moderado"
        else:
            faixa_risco = "Alto Risco"

        return PredicaoCreditoOutput(
            nome_cliente=cliente.nome_cliente,
            idade_informada=cliente.idade,
            pontuacao_informada=cliente.pontuacao_credito,
            decisao=status_decisao,
            codigo_decisao=resultado_classe,
            probabilidade_aprovacao=round(prob_aprovacao, 4),
            probabilidade_risco_inadimplencia=round(prob_risco, 4),
            faixa_risco=faixa_risco,
            indice_comprometimento_anual=round(indice_comprometimento, 4),
            versao_modelo=self._versao_modelo,
        )


model_service = CreditoModelService()

In [ ]:
%%writefile app/services/__init__.py
from app.services.credito_service import CreditoModelService, model_service

__all__ = ["CreditoModelService", "model_service"]

---
## 🌐 Fase 5: Implementação de Endpoints, `Lifespan` e Tratamento de Erros (`FastAPI & HTTP`)

### Justificativas Técnicas (`app/routers/predict_routes.py` e `main.py`):
- **Monitoramento (`GET /health`)**: Responde `200 OK` informando se o serviço está ativo e se `modelo_carregado_em_ram == True`.
- **Inferência (`POST /predict` e `POST /v1/predict`)**: Consome `DadosClienteInput`, aciona `model_service.realizar_predicao()` e devolve `PredicaoCreditoOutput` (`200 OK`).
- **Semântica HTTP Completa**: `200 OK` (sucesso), `400 Bad Request` (violação de regra financeira), `422 Unprocessable Entity` (erro de validação Pydantic) e `500 Internal Server Error` (tratamento seguro sem queda do servidor).

In [ ]:
%%writefile app/routers/predict_routes.py
from fastapi import APIRouter, HTTPException, status

from app.schemas.cliente import (
    DadosClienteInput,
    HealthResponse,
    PredicaoCreditoOutput,
)
from app.services.credito_service import (
    ModeloNaoCarregadoException,
    RegraNegocioException,
    model_service,
)

health_router = APIRouter(tags=["Monitoramento e Saude"])
predict_router = APIRouter(tags=["Predicao de Credito"])


@health_router.get(
    "/health",
    response_model=HealthResponse,
    status_code=status.HTTP_200_OK,
    summary="Verifica a saude da API e se o modelo reside em memoria RAM",
)
def verificar_saude() -> HealthResponse:
    """Retorna HTTP 200 OK com o estado operacional do servico."""
    status_atual = model_service.obter_status_saude()
    return HealthResponse(**status_atual)


def _executar_fluxo_predicao(
    cliente: DadosClienteInput,
) -> PredicaoCreditoOutput:
    """Invoca a camada de servico e mapeia excecoes para codigos HTTP."""
    try:
        return model_service.realizar_predicao(cliente)
    except RegraNegocioException as erro_negocio:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=str(erro_negocio),
        ) from erro_negocio
    except ModeloNaoCarregadoException as erro_modelo:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Modelo indisponivel em RAM: {str(erro_modelo)}",
        ) from erro_modelo
    except Exception as erro_interno:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Erro interno na predicao: {str(erro_interno)}",
        ) from erro_interno


@predict_router.post(
    "/predict",
    response_model=PredicaoCreditoOutput,
    status_code=status.HTTP_200_OK,
    summary="Avalia o risco de credito e retorna a decisao (Rota Padrao)",
)
def prever_aprovacao(cliente: DadosClienteInput) -> PredicaoCreditoOutput:
    return _executar_fluxo_predicao(cliente)


@predict_router.post(
    "/v1/predict",
    response_model=PredicaoCreditoOutput,
    status_code=status.HTTP_200_OK,
    summary="Avalia o risco de credito e retorna a decisao (Rota v1)",
)
def prever_aprovacao_v1(cliente: DadosClienteInput) -> PredicaoCreditoOutput:
    return _executar_fluxo_predicao(cliente)

In [ ]:
%%writefile app/routers/__init__.py
from app.routers.predict_routes import health_router, predict_router

__all__ = ["health_router", "predict_router"]

In [ ]:
%%writefile main.py
from contextlib import asynccontextmanager
from fastapi import FastAPI

from app.routers.predict_routes import health_router, predict_router
from app.services.credito_service import model_service
from train_pipeline import executar_treinamento


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Carrega o modelo em memoria RAM 1 unica vez no startup da API."""
    if not model_service.model_path.exists():
        print("[LIFESPAN] Treinando pipeline inicial automaticamente...")
        executar_treinamento()

    model_service.carregar_modelo_em_ram()
    app.state.model_service = model_service
    print("[LIFESPAN] Modelo carregado em RAM com sucesso!")

    yield

    print("[LIFESPAN] Liberando modelo da memoria RAM...")
    model_service.descarregar_modelo_da_ram()


app = FastAPI(
    title="API de Predicao de Credito - MLOps",
    description=(
        "Servico preditivo RESTful para aprovacao automatica de emprestimos "
        "com Scikit-Learn Pipeline, Pydantic e Lifespan em memoria RAM."
    ),
    version="1.0.0",
    lifespan=lifespan,
)
app.include_router(health_router)
app.include_router(predict_router)

---
## ✅ Validação Interativa dos Endpoints e Suíte Automatizada (`Pytest`)

Abaixo criamos a suíte `tests/test_api.py` e executamos tanto o `pytest -v` quanto demonstrações ao vivo dos retornos `200 OK`, `422 Unprocessable Entity` e `400 Bad Request` diretamente aqui na célula do Colab.

In [ ]:
%%writefile tests/test_api.py
from fastapi.testclient import TestClient
from main import app


def test_health_endpoint_modelo_em_ram():
    """Verifica se o Lifespan carregou o modelo em RAM (GET /health -> 200)."""
    with TestClient(app) as client:
        response = client.get("/health")
        assert response.status_code == 200
        dados = response.json()
        assert dados["status_servico"] == "online"
        assert dados["modelo_carregado_em_ram"] is True


def test_predict_cliente_aprovado_200():
    """Verifica inferencia de aprovacao com payload completo (200 OK)."""
    payload = {
        "nome_cliente": "Ana Beatriz Souza",
        "idade": 38,
        "pontuacao_credito": 780,
        "renda_mensal": 12000.0,
        "valor_emprestimo": 15000.0,
        "tempo_emprego_anos": 8.0,
        "inadimplencia_previa": 0,
        "tipo_moradia": "PROPRIA",
        "finalidade_emprestimo": "NEGOCIOS",
    }
    with TestClient(app) as client:
        response = client.post("/predict", json=payload)
        assert response.status_code == 200
        assert response.json()["decisao"] == "Aprovado"


def test_predict_v1_cliente_reprovado_200():
    """Verifica inferencia de reprovacao na rota /v1/predict (200 OK)."""
    payload = {
        "nome_cliente": "Roberto Silva",
        "idade": 21,
        "pontuacao_credito": 360,
        "renda_mensal": 1800.0,
        "valor_emprestimo": 45000.0,
        "tempo_emprego_anos": 0.5,
        "inadimplencia_previa": 1,
        "tipo_moradia": "ALUGADA",
        "finalidade_emprestimo": "QUITACAO_DIVIDAS",
    }
    with TestClient(app) as client:
        response = client.post("/v1/predict", json=payload)
        assert response.status_code == 200
        assert response.json()["decisao"] == "Reprovado"


def test_predict_payload_minimo_retrocompativel_200():
    """Garante retrocompatibilidade com contrato base (idade e score)."""
    with TestClient(app) as client:
        resp = client.post("/predict", json={"idade": 42, "pontuacao_credito": 740})
        assert resp.status_code == 200


def test_predict_erro_validacao_pydantic_422():
    """Garante que valores fora de Field(ge, le) gerem HTTP 422."""
    with TestClient(app) as client:
        resp = client.post("/predict", json={"idade": 15, "pontuacao_credito": 950})
        assert resp.status_code == 422


def test_predict_erro_regra_negocio_400():
    """Garante que inconsistencia de negocio gere HTTP 400 Bad Request."""
    payload = {"idade": 20, "pontuacao_credito": 700, "tempo_emprego_anos": 15.0}
    with TestClient(app) as client:
        resp = client.post("/predict", json=payload)
        assert resp.status_code == 400

In [ ]:
# 1. Executa a suite Pytest completa
!pytest -v

# 2. Demonstracao ao vivo dos payloads e codigos HTTP para apresentacao ao Mentor
import json
from fastapi.testclient import TestClient
from main import app

with TestClient(app) as client:
    print("\n--- [1] TESTE GET /health (HTTP 200 OK) ---")
    r_health = client.get("/health")
    print(f"Status: {r_health.status_code}")
    print(json.dumps(r_health.json(), indent=2, ensure_ascii=False))

    print("\n--- [2] TESTE POST /predict CLIENTE BAIXO RISCO (HTTP 200 OK) ---")
    r_ok = client.post(
        "/predict",
        json={
            "nome_cliente": "Carlos Eduardo Santos",
            "idade": 35,
            "pontuacao_credito": 720,
            "renda_mensal": 7500.0,
            "valor_emprestimo": 18000.0,
            "tempo_emprego_anos": 5.0,
            "inadimplencia_previa": 0,
            "tipo_moradia": "PROPRIA",
            "finalidade_emprestimo": "NEGOCIOS",
        },
    )
    print(f"Status: {r_ok.status_code}")
    print(json.dumps(r_ok.json(), indent=2, ensure_ascii=False))

    print("\n--- [3] TESTE BLOQUEIO PYDANTIC FIELD (HTTP 422 UNPROCESSABLE ENTITY) ---")
    r_422 = client.post("/predict", json={"idade": 15, "pontuacao_credito": 990})
    print(f"Status: {r_422.status_code}")
    print(json.dumps(r_422.json(), indent=2, ensure_ascii=False))

    print("\n--- [4] TESTE VIOLACAO REGRA DE NEGOCIO (HTTP 400 BAD REQUEST) ---")
    r_400 = client.post(
        "/predict",
        json={"idade": 20, "pontuacao_credito": 700, "tempo_emprego_anos": 15.0},
    )
    print(f"Status: {r_400.status_code}")
    print(json.dumps(r_400.json(), indent=2, ensure_ascii=False))

---
## 📊 Dashboard Executivo em Streamlit (`dashboard.py`) + Execução de Servidores no Colab

A célula abaixo grava o arquivo `dashboard.py` completo e a célula seguinte inicia o servidor **FastAPI (`Uvicorn` na porta `8000`)** em segundo plano dentro do próprio Google Colab, permitindo também expor o **Swagger (`/docs`)** ou o **Streamlit (`8501`)**.

In [ ]:
%%writefile dashboard.py
from pathlib import Path
import json
import time
from typing import Any, Dict, Tuple
import httpx
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

BASE_DIR = Path(__file__).resolve().parent
DATASET_PATH = BASE_DIR / "data" / "credito_dataset.csv"
METRICS_PATH = BASE_DIR / "models" / "relatorio_metricas.json"

st.set_page_config(
    page_title="CreditAI MLOps | Dashboard & Simulador FastAPI",
    page_icon="🏦",
    layout="wide",
)


def consultar_health_api(
    base_url: str,
) -> Tuple[bool, int, Dict[str, Any], float]:
    """Chama GET /health na API FastAPI e mede a latencia em ms."""
    inicio = time.perf_counter()
    try:
        with httpx.Client(timeout=4.0) as client:
            resp = client.get(f"{base_url.rstrip('/')}/health")
            latencia_ms = (time.perf_counter() - inicio) * 1000.0
            return (
                resp.status_code == 200,
                resp.status_code,
                resp.json(),
                latencia_ms,
            )
    except Exception as exc:
        latencia_ms = (time.perf_counter() - inicio) * 1000.0
        return False, 0, {"erro_conexao": str(exc)}, latencia_ms


def chamar_predict_api(
    base_url: str,
    endpoint: str,
    payload: Dict[str, Any],
) -> Tuple[int, Dict[str, Any], float]:
    """Envia POST /predict ou /v1/predict para a API FastAPI."""
    inicio = time.perf_counter()
    url = f"{base_url.rstrip('/')}/{endpoint.lstrip('/')}"
    try:
        with httpx.Client(timeout=5.0) as client:
            resp = client.post(url, json=payload)
            latencia_ms = (time.perf_counter() - inicio) * 1000.0
            return resp.status_code, resp.json(), latencia_ms
    except Exception as exc:
        latencia_ms = (time.perf_counter() - inicio) * 1000.0
        return 503, {"detail": f"API offline: {str(exc)}"}, latencia_ms


def criar_grafico_velocimetro(prob: float, decisao: str) -> go.Figure:
    """Cria um velocimetro (Gauge) Plotly para a probabilidade."""
    cor_barra = "#10b981" if decisao == "Aprovado" else "#ef4444"
    fig = go.Figure(
        go.Indicator(
            mode="gauge+number",
            value=prob * 100.0,
            number={"suffix": "%"},
            title={"text": "Probabilidade de Aprovacao (Modelo em RAM)"},
            gauge={"axis": {"range": [0, 100]}, "bar": {"color": cor_barra}},
        )
    )
    return fig


st.title("🏦 CreditAI MLOps — Dashboard Executivo & Simulador FastAPI")
api_base_url = st.sidebar.text_input("URL Base FastAPI", value="http://127.0.0.1:8000")
rota_escolhida = st.sidebar.selectbox("Endpoint", ["/predict", "/v1/predict"])

online, sc_h, dados_h, lat_h = consultar_health_api(api_base_url)
st.sidebar.metric(
    "Status GET /health",
    f"ONLINE ({sc_h})" if online else "OFFLINE",
    f"{lat_h:.1f} ms",
)

aba_simulador, aba_governanca, aba_analytics = st.tabs([
    "1. Simulador Preditivo de Credito (POST /predict)",
    "2. Laboratorio de Validacao & Erros HTTP (200 / 400 / 422)",
    "3. Metricas do Pipeline & Exploracao de Dados (Fase 1)",
])

with aba_simulador:
    c1, c2 = st.columns(2)
    with c1:
        nome_cliente = st.text_input("Nome do Solicitante", value="Carlos Eduardo Santos")
        idade = st.slider("Idade", 18, 100, 35)
        pontuacao_credito = st.slider("Score de Credito", 300, 850, 720)
        renda_mensal = st.number_input("Renda Mensal (R$)", 100.0, 1000000.0, 7500.0)
        valor_emprestimo = st.number_input("Valor Solicitado (R$)", 500.0, 500000.0, 18000.0)
        tempo_emprego_anos = st.number_input("Tempo de Emprego (Anos)", 0.0, 60.0, 5.0)
        inadimplencia_previa = st.selectbox("Inadimplencia Previa", [0, 1])
        tipo_moradia = st.selectbox("Moradia", ["PROPRIA", "ALUGADA", "FINANCIADA", "OUTROS"])
        finalidade_emprestimo = st.selectbox(
            "Finalidade",
            ["EDUCACAO", "SAUDE", "NEGOCIOS", "PESSOAL", "QUITACAO_DIVIDAS", "IMOVEL"],
        )
    with c2:
        payload_envio = {
            "nome_cliente": nome_cliente,
            "idade": int(idade),
            "pontuacao_credito": int(pontuacao_credito),
            "renda_mensal": float(renda_mensal),
            "valor_emprestimo": float(valor_emprestimo),
            "tempo_emprego_anos": float(tempo_emprego_anos),
            "inadimplencia_previa": int(inadimplencia_previa),
            "tipo_moradia": tipo_moradia,
            "finalidade_emprestimo": finalidade_emprestimo,
        }
        status_code, resposta_json, latencia_pred = chamar_predict_api(
            api_base_url, rota_escolhida, payload_envio
        )
        if status_code == 200:
            st.success(f"Decisao: {resposta_json['decisao']} ({resposta_json['faixa_risco']})")
            fig_gauge = criar_grafico_velocimetro(
                resposta_json["probabilidade_aprovacao"],
                resposta_json["decisao"],
            )
            st.plotly_chart(fig_gauge, use_container_width=True)
        st.json(resposta_json)

with aba_governanca:
    if st.button("Testar Contrato Minimo (200 OK)"):
        sc, resp, lat = chamar_predict_api(
            api_base_url, rota_escolhida, {"idade": 40, "pontuacao_credito": 760}
        )
        st.success(f"Status HTTP: {sc} OK ({lat:.1f} ms)")
        st.json(resp)
    if st.button("Testar Bloqueio Pydantic (422)"):
        sc, resp, lat = chamar_predict_api(
            api_base_url, rota_escolhida, {"idade": 15, "pontuacao_credito": 990}
        )
        st.warning(f"Status HTTP: {sc} Unprocessable Entity")
        st.json(resp)
    if st.button("Testar Regra Financeira (400)"):
        sc, resp, lat = chamar_predict_api(
            api_base_url,
            rota_escolhida,
            {"idade": 20, "pontuacao_credito": 710, "tempo_emprego_anos": 14.0},
        )
        st.error(f"Status HTTP: {sc} Bad Request")
        st.json(resp)

with aba_analytics:
    if METRICS_PATH.exists():
        relatorio = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
        st.json(relatorio["metricas"])

In [ ]:
# Inicia o servidor FastAPI (Uvicorn) em background dentro do Google Colab / Jupyter
# e realiza uma chamada HTTP real com HTTPX na porta 8000
import subprocess
import time
import httpx

proc_uvicorn = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(3)

try:
    with httpx.Client() as client:
        r_health = client.get("http://127.0.0.1:8000/health")
        r_pred = client.post(
            "http://127.0.0.1:8000/v1/predict",
            json={"idade": 36, "pontuacao_credito": 750},
        )
        print("[SERVIDOR UVICORN ATIVO EM http://127.0.0.1:8000]")
        print("Resposta GET /health ->", r_health.status_code, r_health.json()["status_servico"])
        print("Resposta POST /v1/predict ->", r_pred.status_code, r_pred.json())
finally:
    proc_uvicorn.terminate()

---
## 🔀 Seção 5.3: Automação do Versionamento GitFlow no GitHub

A célula abaixo executa automaticamente a estrutura de branches exigida nos **Critérios de Avaliação (Seção 5.3 e Bloco 2 da Rubrica)**:
- Cria a branch `main` e a branch `develop`.
- Cria uma **feature branch separada para cada etapa** (`feature/fase1-pipeline-ml`, `feature/fase4-schemas-pydantic`, `feature/fase2-3-service-lifespan`, `feature/fase5-endpoints-dashboard`), realizando commits concisos no presente do indicativo.
- Realiza os merges na `develop` **sem excluir nenhuma feature branch** (`--no-ff`) e consolida a entrega na `main`.

In [ ]:
%%bash
# Configuracao de usuario Git local (ajuste com seu nome e e-mail do GitHub)
git config --global user.name "Estudante SCTEC"
git config --global user.email "estudante@sctec.edu.br"

# Remove .git anterior caso a celula seja re-executada no Colab
rm -rf .git

# 1. Inicializa branch main
git init -b main
git add requirements.txt
git commit -m "adiciona arquivo requirements.txt com dependencias fixadas"

# 2. Cria branch develop
git checkout -b develop

# 3. Feature Branch Fase 1
git checkout -b feature/fase1-pipeline-ml develop
git add train_pipeline.py data/ models/
git commit -m "implementa pipeline de machine learning e serializa modelo"
git checkout develop
git merge --no-ff feature/fase1-pipeline-ml -m "integra feature/fase1-pipeline-ml na develop"

# 4. Feature Branch Fase 4
git checkout -b feature/fase4-schemas-pydantic develop
git add app/__init__.py app/schemas/
git commit -m "implementa schemas pydantic de entrada e saida com validacao field"
git checkout develop
git merge --no-ff feature/fase4-schemas-pydantic -m "integra feature/fase4-schemas-pydantic na develop"

# 5. Feature Branch Fases 2 e 3
git checkout -b feature/fase2-3-service-lifespan develop
git add app/services/
git commit -m "implementa camada de servico para inferencia em memoria ram"
git checkout develop
git merge --no-ff feature/fase2-3-service-lifespan -m "integra feature/fase2-3-service-lifespan na develop"

# 6. Feature Branch Fase 5 e Dashboard
git checkout -b feature/fase5-endpoints-dashboard develop
git add app/routers/ main.py dashboard.py tests/
git commit -m "implementa endpoints fastapi com lifespan, dashboard streamlit e testes"
git checkout develop
git merge --no-ff feature/fase5-endpoints-dashboard -m "integra feature/fase5-endpoints-dashboard na develop"

# 7. Merge final na main
git checkout main
git merge --no-ff develop -m "consolida versao final 1.0.0 na branch main"

echo ""
echo "=== HISTORICO DE BRANCHES E COMMITS (GITFLOW COMPLETO) ==="
git branch -a
git log --oneline --graph --all